Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# OWL-v2 Zero-Shot Inference on Coffee & Cashew Datasets

**Group 1 — Open-Vocabulary Object Detection**

This notebook runs zero-shot object detection using [OWL-v2](https://huggingface.co/google/owlv2-base-patch16-ensemble) on the Coffee & Cashew Nut dataset. The model uses **text prompts** instead of class IDs — no training required.

**What this notebook does:**
1. Downloads the dataset and prepares COCO ground truth
2. Runs OWL-v2 inference using text prompts
3. Converts predictions to COCO format
4. Evaluates with COCOEval (AP@50) + threshold sweep
5. Visualises predictions vs. ground truth

**How to run:** Click **Runtime → Run all**, or step through cell by cell.
Set `NUM_EXAMPLES = 4` for a quick smoke test (~2 min on T4).

## 1. Setup

In [ ]:
# @title 1a. Install dependencies
!pip install -q transformers torch torchvision pycocotools Pillow matplotlib

In [ ]:
# @title 1b. Clone the repo (for common scripts)
import os

REPO_DIR = "google-research/betum_tool"

if not os.path.exists("google-research"):
    repo_url = "https://github.com/google-research/google-research.git"
    print("Cloning google-research monorepo (sparse checkout)...")
    # We do a sparse checkout of only the betum_tool directory to avoid downloading 2GB+ of monorepo code.
    !git clone --depth=1 --no-checkout {repo_url} google-research
    !cd google-research && git sparse-checkout set betum_tool && git checkout
    print("✓ Cloned google-research/betum_tool/")
else:
    print("Repo already cloned.")

# Verify common scripts exist
assert os.path.exists(f"{REPO_DIR}/common/yolo_to_coco.py"), "Missing yolo_to_coco.py"
assert os.path.exists(f"{REPO_DIR}/common/class_map.json"), "Missing class_map.json"
print("✓ Common scripts found.")


In [ ]:
# @title 1c. Download dataset from Mendeley
import os
import subprocess
import requests

DATA_DIR = "data"
MENDELEY_URL = "https://data.mendeley.com/public-api/zip/r46c6bpfpf/download/1"
ZIP_NAME = "dataset.zip"

os.makedirs(DATA_DIR, exist_ok=True)

zip_path = os.path.join(DATA_DIR, ZIP_NAME)
if not os.path.exists(zip_path):
    print("Downloading dataset from Mendeley (this may take a few minutes)...")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        print("Attempting download with requests...")
        response = requests.get(MENDELEY_URL, headers=headers, stream=True)
        if response.status_code == 200:
            with open(zip_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"Downloaded to {zip_path}")
        else:
            print(f"HTTP Error {response.status_code} with requests.")
            raise Exception(f"HTTP Error {response.status_code}")
            
    except Exception as e:
        print(f"Requests failed: {e}")
        print("Trying fallback with wget...")
        try:
            subprocess.run(["wget", "-U", "Mozilla/5.0", "-O", zip_path, MENDELEY_URL], check=True)
            print(f"Downloaded with wget to {zip_path}")
        except Exception as wget_e:
            print(f"Wget also failed: {wget_e}")
            print("Please manually download the dataset from https://data.mendeley.com/datasets/r46c6bpfpf/1 and upload it to the 'data' folder.")
else:
    print(f"Dataset already downloaded at {zip_path}")

In [ ]:
# @title 1d. Unzip main archive
import zipfile
import os

DATA_DIR = "data"
ZIP_NAME = "dataset.zip"
zip_path = os.path.join(DATA_DIR, ZIP_NAME)

print("Unzipping main archive...")
try:
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    print("Done!")
except Exception as e:
    print(f"Error unzipping {zip_path}: {e}")

# Debug: List contents to see what we got
print("Contents of DATA_DIR after unzip:")
for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}[{os.path.basename(root)}/]")
    subindent = ' ' * 4 * (level + 1)
    for f in files[:5]: # Show first 5 files to avoid spam
        print(f"{subindent}{f}")
    if len(files) > 5:
        print(f"{subindent}... ({len(files) - 5} more files)")

In [ ]:
# @title 1e. Extract .rar files
import os
import subprocess

DATA_DIR = "data"

print("Searching for .rar files...")
rar_files = []
for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.endswith(".rar"):
            rar_files.append(os.path.join(root, file))

print(f"Found rar files: {rar_files}")

for rar_path in rar_files:
    print(f"Extracting {rar_path}...")
    try:
        # Extract directly to DATA_DIR so that Cashew and Coffee folders 
        # end up in data/ instead of data/Coffee and Cashew Nut Dataset/
        subprocess.run(["unrar", "x", "-o+", rar_path, DATA_DIR], check=True)
        print(f"Extracted {rar_path} to {DATA_DIR}")
    except Exception as e:
        print(f"Error extracting {rar_path}: {e}")
        print("Please ensure 'unrar' is installed (e.g., '!apt-get install unrar' in a cell).")

In [ ]:
# @title 1f. Verify dataset structure
import os

DATA_DIR = "data"

print("Verifying structure...")
for expected in ["Cashew/Cashew-Uganda/images", "Coffee/Batch1/images"]:
    found = False
    for root, dirs, files in os.walk(DATA_DIR):
        if expected in root:
            found = True
            print(f"✓ Found {expected} at {root}")
            break
    if not found:
        print(f"Warning: Could not find expected directory structure containing '{expected}'")
        
print("✓ Verification check complete.")

In [ ]:
# @title 1g. Flatten Coffee + Convert YOLO → COCO JSON
import shutil

# --- Flatten Coffee batches ---
# flatten_coffee.py expects data/Coffee/ in the current directory
coffee_flat_dir = os.path.join(DATA_DIR, "Coffee_flattened")
if os.path.exists(coffee_flat_dir):
    print("Cleaning up existing Coffee_flattened directory to force re-flattening...")
    shutil.rmtree(coffee_flat_dir)

print("Flattening Coffee dataset...")
!python {REPO_DIR}/common/flatten_coffee.py

# --- Convert YOLO → COCO for Cashew ---
os.makedirs(os.path.join(DATA_DIR, "coco"), exist_ok=True)

cashew_coco = os.path.join(DATA_DIR, "coco", "cashew_val.json")
if not os.path.exists(cashew_coco):
    !python {REPO_DIR}/common/yolo_to_coco.py \
        --images {DATA_DIR}/Cashew/Cashew-Uganda/images \
        --labels {DATA_DIR}/Cashew/Cashew-Uganda/Labels \
        --class_map {REPO_DIR}/common/class_map.json \
        --dataset cashew \
        --output {DATA_DIR}/coco/ \
        --split_ratio 0.8
else:
    print("Cashew COCO JSON already exists.")

# --- Convert YOLO → COCO for Coffee ---
coffee_coco = os.path.join(DATA_DIR, "coco", "coffee_val.json")
if not os.path.exists(coffee_coco):
    !python {REPO_DIR}/common/yolo_to_coco.py \
        --images {DATA_DIR}/Coffee_flattened/images \
        --labels {DATA_DIR}/Coffee_flattened/labels \
        --class_map {REPO_DIR}/common/class_map.json \
        --dataset coffee \
        --output {DATA_DIR}/coco/ \
        --split_ratio 0.8
else:
    print("Coffee COCO JSON already exists.")

print("\n✓ COCO ground truth ready.")

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from transformers import Owlv2ForObjectDetection, Owlv2Processor

print(f"torch version: {torch.__version__}")

## 2. Configuration

Adjust these variables for your setup. Set `NUM_EXAMPLES = 4` for a quick smoke test, or `None` for a full run.

In [ ]:
# @title Configuration {display-mode: "form"}

# --- Smoke test ---
NUM_EXAMPLES = 4  # Set to None for full dataset

# --- Dataset selection ---
DATASET = "cashew"  # @param ["cashew", "coffee"]

# --- Paths (auto-configured from setup cells above) ---
COCO_VAL_JSON = Path(f"data/coco/{DATASET}_val.json")

IMAGE_DIRS = {
    "cashew": Path("data/Cashew/Cashew-Uganda/images"),
    "coffee": Path("data/Coffee_flattened/images"),
}
IMAGE_DIR = IMAGE_DIRS[DATASET]

# --- Model ---
MODEL_ID = "google/owlv2-base-patch16-ensemble"

# --- Detection ---
CONFIDENCE_THRESHOLD = 0.01  # Start low for zero-shot; tune later
SCORE_THRESHOLDS_TO_SWEEP = [0.01, 0.02, 0.03, 0.04, 0.05]

# --- Text prompts per dataset ---
# Order matches category_ids 0, 1, 2, ... in the COCO JSON.
# ⚡ Experiment with these! Prompt engineering is key for zero-shot.
TEXT_QUERIES = {
    "cashew": [
        "a cashew tree",
        "a cashew flower",
        "a premature cashew nut",
        "an unripe cashew nut",
        "a ripe cashew nut",
        "a spoilt cashew nut",
    ],
    "coffee": [
        "an unripe coffee cherry",
        "a ripening coffee cherry",
        "a ripe coffee cherry",
        "a spoilt coffee cherry",
        "a coffee tree",
    ],
}

print(f"Dataset: {DATASET}")
print(f"NUM_EXAMPLES: {NUM_EXAMPLES}")
print(f"COCO JSON: {COCO_VAL_JSON}")
print(f"Image dir: {IMAGE_DIR}")
print(f"Text queries: {TEXT_QUERIES[DATASET]}")

## 3. Device Selection

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## 4. Load Data

Load the COCO ground truth JSON to get the image list and annotations for evaluation.

In [ ]:
# Load COCO ground truth
with open(COCO_VAL_JSON, "r") as f:
    coco_gt_data = json.load(f)

images_info = coco_gt_data["images"]
categories = {cat["id"]: cat["name"] for cat in coco_gt_data["categories"]}

# Limit to NUM_EXAMPLES if set
if NUM_EXAMPLES is not None:
    images_info = images_info[:NUM_EXAMPLES]

print(f"Loaded {len(images_info)} images from {COCO_VAL_JSON.name}")
print(f"Categories: {categories}")

# Build query index → category_id mapping
# Text queries are ordered to match category IDs 0, 1, 2, ...
queries = TEXT_QUERIES[DATASET]
query_index_to_category_id = {i: i for i in range(len(queries))}
print(f"\nQuery → Category mapping:")
for qi, cat_id in query_index_to_category_id.items():
    print(f"  [{qi}] \"{queries[qi]}\" → category_id {cat_id} ({categories[cat_id]})")

## 5. Load Model

Load OWL-v2 from HuggingFace Hub. First run will download the weights (~600 MB).

In [ ]:
processor = Owlv2Processor.from_pretrained(MODEL_ID)
model = Owlv2ForObjectDetection.from_pretrained(MODEL_ID).to(device)
model.eval()

print(f"Model loaded: {MODEL_ID}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 6. Run Inference

Process images one at a time (batch=1 for simplicity). For each image:
1. Load the image
2. Run the processor with text queries
3. Forward pass through the model
4. Post-process: extract boxes, scores, labels above the confidence threshold

In [ ]:
def run_inference(
    model, processor, image, text_queries, device, score_threshold=0.1
):
    """Run OWL-v2 inference on a single image.

    Args:
        model: Owlv2ForObjectDetection model.
        processor: Owlv2Processor.
        image: PIL Image.
        text_queries: List of text prompts.
        device: torch device.
        score_threshold: Minimum confidence to keep a detection.

    Returns:
        Dict with 'boxes' (list of [cx,cy,w,h] normalised),
        'scores' (list of float), 'labels' (list of int query indices).
    """
    inputs = processor(
        text=text_queries, images=image, return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # OWL-v2 outputs: logits [1, num_queries * num_patches, num_classes]
    #                 pred_boxes [1, num_queries * num_patches, 4] in [cx,cy,w,h]
    logits = outputs.logits[0]  # [N, num_classes]
    pred_boxes = outputs.pred_boxes[0]  # [N, 4]

    # Sigmoid → scores, then take max across classes
    probs = logits.sigmoid()
    scores, labels = probs.max(dim=-1)

    # Filter by threshold
    mask = scores > score_threshold
    filtered_boxes = pred_boxes[mask].cpu().tolist()
    filtered_scores = scores[mask].cpu().tolist()
    filtered_labels = labels[mask].cpu().tolist()

    return {
        "boxes": filtered_boxes,
        "scores": filtered_scores,
        "labels": filtered_labels,
    }

In [ ]:
# Run inference on all images
queries = TEXT_QUERIES[DATASET]
all_predictions = []

for i, img_info in enumerate(images_info):
    img_path = IMAGE_DIR / img_info["file_name"]
    if not img_path.exists():
        print(f"  WARNING: Image not found: {img_path}")
        continue

    image = Image.open(img_path).convert("RGB")
    result = run_inference(
        model, processor, image, queries, device,
        score_threshold=CONFIDENCE_THRESHOLD
    )

    all_predictions.append({
        "image_id": img_info["id"],
        "img_width": img_info["width"],
        "img_height": img_info["height"],
        "boxes": result["boxes"],
        "scores": result["scores"],
        "labels": result["labels"],
    })

    if (i + 1) % 10 == 0 or (i + 1) == len(images_info):
        total_dets = sum(len(p["boxes"]) for p in all_predictions)
        print(f"  Processed {i + 1}/{len(images_info)} images "
              f"({total_dets} detections so far)")

total_dets = sum(len(p["boxes"]) for p in all_predictions)
print(f"\nDone! {len(all_predictions)} images, {total_dets} total detections "
      f"(threshold={CONFIDENCE_THRESHOLD})")

## 7. Convert to COCO Format

Convert OWL-v2's normalised `[cx, cy, w, h]` boxes to COCO absolute `[x_min, y_min, width, height]` format.

In [ ]:
def owlv2_box_to_coco(cx, cy, w, h, img_width, img_height):
    """Convert OWL-v2 normalised [cx,cy,w,h] to COCO absolute [xmin,ymin,w,h].

    OWL-v2 normalises by the longest edge, not independently by width/height.
    """
    max_side = max(img_width, img_height)
    abs_cx = cx * max_side
    abs_cy = cy * max_side
    abs_w = w * max_side
    abs_h = h * max_side

    x_min = max(0.0, abs_cx - abs_w / 2)
    y_min = max(0.0, abs_cy - abs_h / 2)
    # Clamp to image bounds
    abs_w = min(abs_w, float(img_width) - x_min)
    abs_h = min(abs_h, float(img_height) - y_min)

    return [round(x_min, 2), round(y_min, 2), round(abs_w, 2), round(abs_h, 2)]


def to_coco_predictions(predictions, query_index_to_category_id):
    """Convert all predictions to COCO format."""
    coco_preds = []
    for pred in predictions:
        img_w = pred["img_width"]
        img_h = pred["img_height"]

        for box, score, label in zip(
            pred["boxes"], pred["scores"], pred["labels"]
        ):
            cat_id = query_index_to_category_id.get(label)
            if cat_id is None:
                continue

            coco_box = owlv2_box_to_coco(
                box[0], box[1], box[2], box[3], img_w, img_h
            )
            coco_preds.append({
                "image_id": pred["image_id"],
                "category_id": cat_id,
                "bbox": coco_box,
                "score": round(float(score), 4),
            })
    return coco_preds


coco_predictions = to_coco_predictions(all_predictions, query_index_to_category_id)
print(f"Generated {len(coco_predictions)} COCO-format predictions")

## 8. Evaluate with COCOEval

Run the standard COCO evaluation to compute AP@50.

In [ ]:
def run_coco_eval(gt_json_path, predictions, score_threshold=None):
    """Run COCOEval and return AP@50.

    Args:
        gt_json_path: Path to COCO ground truth JSON.
        predictions: List of COCO prediction dicts.
        score_threshold: If set, filter predictions below this score.

    Returns:
        Dict with AP metrics.
    """
    if score_threshold is not None:
        predictions = [p for p in predictions if p["score"] >= score_threshold]

    if not predictions:
        print("  No predictions above threshold — AP = 0")
        return {"AP": 0.0, "AP@50": 0.0, "AP@75": 0.0}

    coco_gt = COCO(str(gt_json_path))
    coco_dt = coco_gt.loadRes(predictions)

    coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    return {
        "AP": round(coco_eval.stats[0], 4),
        "AP@50": round(coco_eval.stats[1], 4),
        "AP@75": round(coco_eval.stats[2], 4),
    }


# Single evaluation at default threshold
print(f"=== Evaluation at threshold {CONFIDENCE_THRESHOLD} ===\n")
results = run_coco_eval(COCO_VAL_JSON, coco_predictions)
print(f"\nAP@50 = {results['AP@50']}")

## 9. Threshold Sweep

Sweep confidence thresholds to find the sweet spot. Higher thresholds reduce false positives but may miss detections.

In [ ]:
print("=== Threshold Sweep ===\n")
sweep_results = {}

for thresh in SCORE_THRESHOLDS_TO_SWEEP:
    filtered = [p for p in coco_predictions if p["score"] >= thresh]
    print(f"\n--- Threshold {thresh} ({len(filtered)} predictions) ---")
    if filtered:
        res = run_coco_eval(COCO_VAL_JSON, filtered)
        sweep_results[thresh] = res
    else:
        print("  No predictions above threshold.")
        sweep_results[thresh] = {"AP": 0.0, "AP@50": 0.0, "AP@75": 0.0}

# Summary table
print("\n\n=== Summary ===")
print(f"{'Threshold':<12} {'Predictions':<14} {'AP@50':<10}")
print("-" * 36)
for thresh in SCORE_THRESHOLDS_TO_SWEEP:
    n_preds = len([p for p in coco_predictions if p["score"] >= thresh])
    ap50 = sweep_results[thresh]["AP@50"]
    print(f"{thresh:<12} {n_preds:<14} {ap50:<10}")

## 10. Visualise Predictions

Show predictions vs. ground truth side-by-side on sample images.

**Note on Visualization Threshold:**
Object detection models like OWL-v2 output a large number of bounding boxes with associated confidence scores. To prevent the images from being cluttered with low-confidence background noise or weak detections, we use a `VIS_THRESHOLD` (default `0.3`) in the code cell below to filter out predictions with low scores.
- **Higher thresholds** (e.g., `0.5`) will show fewer, more confident boxes (high precision).
- **Lower thresholds** (e.g., `0.1`) will show more boxes (high recall) but may increase clutter and false positives.

You can adjust the `VIS_THRESHOLD` variable in the code cell below to explore how it affects the visualization!

In [ ]:
VIS_THRESHOLD = 0.3  # Threshold for visualisation
NUM_VIS = min(4, len(images_info))  # Number of images to visualise

# Build a colour map for categories
cmap = plt.cm.get_cmap("tab10")
cat_colors = {cat_id: cmap(i % 10) for i, cat_id in enumerate(categories.keys())}

# Group GT annotations by image_id
gt_by_image = {}
for ann in coco_gt_data["annotations"]:
    img_id = ann["image_id"]
    if img_id not in gt_by_image:
        gt_by_image[img_id] = []
    gt_by_image[img_id].append(ann)

# Group predictions by image_id
pred_by_image = {}
for pred in coco_predictions:
    if pred["score"] < VIS_THRESHOLD:
        continue
    img_id = pred["image_id"]
    if img_id not in pred_by_image:
        pred_by_image[img_id] = []
    pred_by_image[img_id].append(pred)


def draw_boxes(ax, boxes_data, is_gt=True):
    """Draw bounding boxes on an axis."""
    for item in boxes_data:
        bbox = item["bbox"]  # [x, y, w, h]
        cat_id = item["category_id"]
        color = cat_colors.get(cat_id, "red")
        linestyle = "-" if is_gt else "--"
        linewidth = 2 if is_gt else 1.5

        rect = patches.Rectangle(
            (bbox[0], bbox[1]), bbox[2], bbox[3],
            linewidth=linewidth, edgecolor=color,
            facecolor="none", linestyle=linestyle,
        )
        ax.add_patch(rect)

        label = categories.get(cat_id, f"cls{cat_id}")
        if not is_gt and "score" in item:
            label = f"{label} {item['score']:.2f}"
        ax.text(
            bbox[0], bbox[1] - 4, label,
            fontsize=7, color=color,
            bbox=dict(boxstyle="round,pad=0.15", facecolor="black", alpha=0.6),
        )


fig, axes = plt.subplots(NUM_VIS, 2, figsize=(16, 5 * NUM_VIS))
if NUM_VIS == 1:
    axes = axes.reshape(1, -1)

for row, img_info in enumerate(images_info[:NUM_VIS]):
    img_path = IMAGE_DIR / img_info["file_name"]
    image = Image.open(img_path).convert("RGB")
    img_id = img_info["id"]

    # Ground truth
    axes[row, 0].imshow(image)
    axes[row, 0].set_title(f"Ground Truth — {img_info['file_name']}", fontsize=10)
    axes[row, 0].axis("off")
    gt_anns = gt_by_image.get(img_id, [])
    draw_boxes(axes[row, 0], gt_anns, is_gt=True)

    # Predictions
    axes[row, 1].imshow(image)
    axes[row, 1].set_title(
        f"OWL-v2 Predictions (threshold={VIS_THRESHOLD})", fontsize=10
    )
    axes[row, 1].axis("off")
    pred_anns = pred_by_image.get(img_id, [])
    draw_boxes(axes[row, 1], pred_anns, is_gt=False)

plt.tight_layout()
plt.show()

## 11. Save Predictions

Save the COCO-format predictions to disk for later comparison with fine-tuned results.

In [ ]:
output_dir = Path("results")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / f"owlv2_zeroshot_{DATASET}_predictions.json"
with open(output_path, "w") as f:
    json.dump(coco_predictions, f, indent=2)

print(f"Saved {len(coco_predictions)} predictions to {output_path}")
print(f"\nDone! Best AP@50 from sweep:")
best_thresh = max(sweep_results, key=lambda t: sweep_results[t]["AP@50"])
print(f"  Threshold={best_thresh}, AP@50={sweep_results[best_thresh]['AP@50']}")